In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos_bolt.circuitlens import CircuitLensBolt
from tsfm_lens.dataset import GiftEvalDataset

In [ ]:
device = "cuda:3" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
GIFT_EVAL_DIR = os.path.join(DATA_DIR, "gift-eval")
plot_save_dir = os.path.join("../figs", "finetune")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
pipeline = CircuitLensBolt.from_pretrained("amazon/chronos-bolt-base", device_map=device, dtype=torch.bfloat16)
num_layers = pipeline.model.config.num_decoder_layers
num_heads = pipeline.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

## Approach

The Chronos Bolt forward pass is:
```
input_patch_embedding → encoder → decoder (12 layers) → output_patch_embedding → quantile forecasts
```

**Goal:** Remove the second half of the decoder layers (layers 6–11), then fine-tune the
`output_patch_embedding` to compensate for the truncated representations.

We do this **end-to-end** rather than caching layer-6 activations and training offline. End-to-end
ensures zero train/test mismatch — the output head sees exactly the same truncated-decoder
representations during training and inference. It also makes it easy to optionally unfreeze
additional components (e.g. the last remaining decoder layer) if the output head alone isn't
expressive enough.

### Baseline predictions (full 12-layer model)

Before truncating, run the full model on a sample so we can compare later.

In [ ]:
system_name = "m4_monthly"
# NOTE: "long" (multiplier=15, pred_len=270) causes assertion errors in GluonTS
# because some M4 monthly series are too short for the required test windows.
# Use "short" (pred_len=18) which fits within the model's native 64-step horizon.
term = "short"

dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=False, data_dir=GIFT_EVAL_DIR)
print(f"Dataset: {system_name}")
print(f"Frequency: {dataset.freq}")
print(f"Prediction length: {dataset.prediction_length}")
print(f"Evaluation windows: {dataset.windows}")

# Collect all (context, future) pairs from the test split
contexts = []
futures = []
for inp, label in dataset.test_data:
    ctx = np.asarray(inp["target"], dtype=np.float32)
    fut = np.asarray(label["target"], dtype=np.float32)
    contexts.append(ctx)
    futures.append(fut)

print(f"Total samples: {len(contexts)}")
print(f"Context lengths: min={min(len(c) for c in contexts)}, max={max(len(c) for c in contexts)}")
print(f"Future length: {futures[0].shape}")

In [ ]:
# Run baseline predictions with the full 12-layer model on a handful of samples
n_preview = 5
baseline_preds = []

with torch.no_grad():
    for i in range(n_preview):
        ctx = torch.tensor(contexts[i]).unsqueeze(0).to(device)
        pred, _ = pipeline.predict_with_full_outputs(ctx, prediction_length=dataset.prediction_length)
        baseline_preds.append(pred.cpu())

### Truncate the decoder

Remove layers 6–11 from the decoder (keep the first 6 layers). Then freeze everything
except `output_patch_embedding`.

In [ ]:
model = pipeline.model
keep_layers = 6

print(f"Original decoder layers: {len(model.decoder.block)}")
model.decoder.block = model.decoder.block[:keep_layers]
print(f"Truncated decoder layers: {len(model.decoder.block)}")

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the output_patch_embedding
for param in model.output_patch_embedding.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print("\noutput_patch_embedding structure:")
print(model.output_patch_embedding)

### Sanity check: truncated model predictions (before fine-tuning)

The truncated model should produce degraded forecasts since the output head was trained
to read layer-12 representations but is now receiving layer-6 representations.

In [ ]:
truncated_preds_before = []

with torch.no_grad():
    for i in range(n_preview):
        ctx = torch.tensor(contexts[i]).unsqueeze(0).to(device)
        pred, _ = pipeline.predict_with_full_outputs(ctx, prediction_length=dataset.prediction_length)
        truncated_preds_before.append(pred.cpu())

### Prepare training data

Build context/target tensor pairs from the dataset. The model's `forward()` method
accepts `context` and `target` tensors directly and computes quantile (pinball) loss
internally, so we just need to batch and feed them in.

In [ ]:
# The model's native prediction_length is 64; targets longer than that get truncated,
# shorter ones get zero-padded internally by the forward() method.
model_pred_len = model.chronos_config.prediction_length
print(f"Model native prediction length: {model_pred_len}")
print(f"Dataset prediction length: {dataset.prediction_length}")

# Use an 80/20 train/val split
val_fraction = 0.2
n_total = len(contexts)
n_val = max(1, int(n_total * val_fraction))
n_train = n_total - n_val

# Shuffle indices with a fixed seed for reproducibility
rng = np.random.default_rng(42)
indices = rng.permutation(n_total)
train_indices = indices[:n_train]
val_indices = indices[n_train:]

print(f"Train samples: {n_train}, Val samples: {n_val}")

In [ ]:
def make_batches(indices, batch_size):
    """Yield (context_batch, target_batch) tensors from the given sample indices.

    Context tensors have variable length, so we right-pad each batch to the
    longest context in that batch. The model handles NaN-masking internally.
    Targets are truncated to model_pred_len (64) since that's the model's
    native output horizon.
    """
    rng_batch = np.random.default_rng()
    shuffled = rng_batch.permutation(indices)

    for start in range(0, len(shuffled), batch_size):
        batch_idx = shuffled[start : start + batch_size]

        ctx_list = [contexts[i] for i in batch_idx]
        fut_list = [futures[i][:model_pred_len] for i in batch_idx]

        # Pad contexts to the same length (right-pad with NaN, which the model masks)
        max_ctx_len = max(len(c) for c in ctx_list)
        ctx_padded = np.full((len(ctx_list), max_ctx_len), np.nan, dtype=np.float32)
        for j, c in enumerate(ctx_list):
            ctx_padded[j, : len(c)] = c

        ctx_batch = torch.tensor(ctx_padded, dtype=torch.float32).to(device)
        fut_batch = torch.tensor(np.stack(fut_list), dtype=torch.float32).to(device)

        yield ctx_batch, fut_batch

### Fine-tuning loop

We use the model's built-in `forward()` which computes quantile (pinball) loss.
Only `output_patch_embedding` parameters receive gradients — the encoder, decoder,
and input embedding are all frozen.

In [ ]:
# Hyperparameters
num_epochs = 50
batch_size = 32
learning_rate = 1e-4
weight_decay = 1e-5

optimizer = torch.optim.AdamW(
    model.output_patch_embedding.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Save initial weights so we can compare / reset if needed
init_state = copy.deepcopy(model.output_patch_embedding.state_dict())

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    epoch_train_loss = 0.0
    n_train_batches = 0

    for ctx_batch, fut_batch in make_batches(train_indices, batch_size):
        optimizer.zero_grad()
        output = model(context=ctx_batch, target=fut_batch)
        loss = output.loss
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
        n_train_batches += 1

    scheduler.step()
    avg_train_loss = epoch_train_loss / max(n_train_batches, 1)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    epoch_val_loss = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for ctx_batch, fut_batch in make_batches(val_indices, batch_size):
            output = model(context=ctx_batch, target=fut_batch)
            epoch_val_loss += output.loss.item()
            n_val_batches += 1

    avg_val_loss = epoch_val_loss / max(n_val_batches, 1)
    val_losses.append(avg_val_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:3d}/{num_epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"LR: {scheduler.get_last_lr()[0]:.2e}"
        )

### Training curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, num_epochs + 1), train_losses, label="Train")
ax.plot(range(1, num_epochs + 1), val_losses, label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Quantile Loss")
ax.set_title("Fine-tuning output_patch_embedding (6-layer decoder)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(plot_save_dir, "training_curves.png"), dpi=200, bbox_inches="tight")
plt.show()

### Compare predictions: full model vs. truncated (before) vs. fine-tuned (after)

For each preview sample we show three forecasts:
1. **Full model (12 layers)** — the original baseline
2. **Truncated (6 layers, before fine-tuning)** — expected to be degraded
3. **Truncated + fine-tuned** — should recover some of the baseline quality

In [ ]:
# Generate fine-tuned predictions
model.eval()
finetuned_preds = []

with torch.no_grad():
    for i in range(n_preview):
        ctx = torch.tensor(contexts[i]).unsqueeze(0).to(device)
        pred, _ = pipeline.predict_with_full_outputs(ctx, prediction_length=dataset.prediction_length)
        finetuned_preds.append(pred.cpu())

In [ ]:
fig, axs = plt.subplots(n_preview, 1, figsize=(8, 3 * n_preview))
if n_preview == 1:
    axs = [axs]

# Median quantile index (0.5 is the 5th of 9 quantiles at indices 0-8)
median_idx = 4

for i, ax in enumerate(axs):
    ctx_np = contexts[i]
    fut_np = futures[i]
    ctx_t = np.arange(len(ctx_np))
    fut_t = np.arange(len(ctx_np), len(ctx_np) + len(fut_np))

    # Context + ground truth
    ax.plot(ctx_t[-200:], ctx_np[-200:], color="black", linewidth=1, label="Context")
    ax.plot(fut_t, fut_np, color="black", linewidth=1, linestyle="--", label="Ground Truth")

    # Full model baseline (median)
    baseline = baseline_preds[i].squeeze()
    pred_t = np.arange(len(ctx_np), len(ctx_np) + baseline.shape[-1])
    ax.plot(pred_t, baseline[median_idx], color="tab:blue", linewidth=1.5, label="Full (12L)")

    # Truncated before fine-tuning (median)
    trunc_before = truncated_preds_before[i].squeeze()
    ax.plot(pred_t, trunc_before[median_idx], color="tab:red", linewidth=1.5, label="Truncated (6L)")

    # Fine-tuned (median)
    finetuned = finetuned_preds[i].squeeze()
    ax.plot(pred_t, finetuned[median_idx], color="tab:green", linewidth=1.5, label="Fine-tuned (6L)")

    ax.set_title(f"Sample {i}")
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper left", fontsize=8)

fig.suptitle("Full vs. Truncated vs. Fine-tuned Predictions (Median Quantile)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(plot_save_dir, "prediction_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()

### Quantile calibration check

Plot all 9 quantile forecasts from the fine-tuned model to verify the prediction
intervals remain well-spread and not collapsed.

In [ ]:
quantile_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
colors = plt.cm.viridis(np.linspace(0, 1, len(quantile_levels)))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
sample_idx = 0
ctx_np = contexts[sample_idx]
fut_np = futures[sample_idx]

for ax, preds_tensor, title in [
    (axs[0], baseline_preds[sample_idx], "Full Model (12L)"),
    (axs[1], finetuned_preds[sample_idx], "Fine-tuned (6L)"),
]:
    preds_np = preds_tensor.squeeze().numpy()
    fut_t = np.arange(len(ctx_np), len(ctx_np) + preds_np.shape[-1])

    ax.plot(np.arange(len(ctx_np))[-100:], ctx_np[-100:], color="black", linewidth=1, label="Context")
    ax.plot(
        np.arange(len(ctx_np), len(ctx_np) + len(fut_np)),
        fut_np,
        color="black",
        linewidth=1,
        linestyle="--",
        label="Ground Truth",
    )

    for qi, (q, c) in enumerate(zip(quantile_levels, colors)):
        ax.plot(fut_t, preds_np[qi], color=c, linewidth=1, alpha=0.8, label=f"q={q:.1f}")

    # Shade 10-90% interval
    ax.fill_between(fut_t, preds_np[0], preds_np[8], alpha=0.15, color="tab:blue")

    ax.set_title(title)
    ax.grid(alpha=0.3)

axs[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(plot_save_dir, "quantile_calibration.png"), dpi=200, bbox_inches="tight")
plt.show()

### Evaluate on full validation set

Compute mean quantile loss across all validation samples for a quantitative comparison.

In [ ]:
model.eval()
total_loss = 0.0
n_batches = 0

with torch.no_grad():
    for ctx_batch, fut_batch in make_batches(val_indices, batch_size):
        output = model(context=ctx_batch, target=fut_batch)
        total_loss += output.loss.item()
        n_batches += 1

print(f"Fine-tuned model — Val quantile loss: {total_loss / n_batches:.4f}")
print(f"(Compare with epoch 1 val loss: {val_losses[0]:.4f})")

### Save fine-tuned weights

In [ ]:
save_path = os.path.join(plot_save_dir, "output_patch_embedding_finetuned.pt")
torch.save(model.output_patch_embedding.state_dict(), save_path)
print(f"Saved fine-tuned output_patch_embedding weights to {save_path}")